In [2]:
# 2.Build a basic Convolutional Neural Network using Keras with a Conv2D -> MaxPooling2D -> Flatten -> Dense stack to classify your
# fashion images into their respective categories.


import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

In [1]:
dataset_path = "Fashion-CNN"

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(128, 128),
    batch_size=4,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

validation_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(128, 128),
    batch_size=4,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

print("Classes:", train_data.class_indices)
print("Training Images:", train_data.samples)
print("Validation Images:", validation_data.samples)

model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(128, 128, 3)),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(train_data.num_classes, activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

model.summary()

Found 25 images belonging to 3 classes.
Found 5 images belonging to 3 classes.
Classes: {'.ipynb_checkpoints': 0, 'fashion_dataset': 1, 'test_images': 2}
Training Images: 25
Validation Images: 5


C:\Users\Kirtan\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 126, 126, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 127008)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │      16,257,152 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │             387 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 16,258,435 (62.02 MB)

 Trainable params: 16,258,435 (62.02 MB)

 Non-trainable params: 0 (0.00 B)

In [2]:
# 3.Train your baseline CNN model for 5 epochs and print the training and validation accuracy after each epoch.

history = model.fit(
    train_data,
    validation_data=validation_data, epochs=5)

for i in range(5):
    print(
        f"Epoch {i+1}: "
        f"Training Accuracy = {history.history['accuracy'][i]:.4f}, "
        f"Validation Accuracy = {history.history['val_accuracy'][i]:.4f}"
    )

Epoch 1/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 383ms/step - accuracy: 0.6000 - loss: 8.5866 - val_accuracy: 0.8000 - val_loss: 4.3077
Epoch 2/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 283ms/step - accuracy: 0.7600 - loss: 3.8248 - val_accuracy: 0.2000 - val_loss: 7.9386
Epoch 3/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 275ms/step - accuracy: 0.7600 - loss: 5.2338 - val_accuracy: 0.8000 - val_loss: 10.3794
Epoch 4/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 277ms/step - accuracy: 0.8400 - loss: 4.3168 - val_accuracy: 0.8000 - val_loss: 4.1519
Epoch 5/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 278ms/step - accuracy: 0.8400 - loss: 0.6943 - val_accuracy: 0.4000 - val_loss: 0.7553
Epoch 1: Training Accuracy = 0.6000, Validation Accuracy = 0.8000
Epoch 2: Training Accuracy = 0.7600, Validation Accuracy = 0.2000
Epoch 3: Training Accuracy = 0.7600, Validation Accuracy = 0.8000
Epoch 4: Training Accuracy = 0.8400, Validation Accuracy = 0.8000
Epoch 5: Training Accuracy = 0.8400, Validation Accuracy = 0.4000


In [7]:
# 4.Evaluate your trained model's accuracy on a separate test set of 6-10 unseen fashion images and display which images were misclassified.<br><br><em><strong>Hint:</strong> Use model.predict()
# and compare the predicted label with the actual folder/category.</em>

import os
import numpy as np
from tensorflow.keras.preprocessing import image

class_names = list(train_data.class_indices.keys())

test_path = "Fashion-CNN/test_images"

correct = 0
total = 0
misclassified = []

for actual_class in os.listdir(test_path):

    class_folder = os.path.join(test_path, actual_class)

    if not os.path.isdir(class_folder):
        continue

    for filename in os.listdir(class_folder):

        image_path = os.path.join(class_folder, filename)

        img = image.load_img(
            image_path,
            target_size=(128, 128)
        )

        img_array = image.img_to_array(img)
        img_array = img_array / 255.0
        img_array = np.expand_dims(img_array, axis=0)

        prediction = model.predict(img_array, verbose=0)

        predicted_class = class_names[np.argmax(prediction)]

        total += 1

        print(
            filename,
            "| Actual:", actual_class,
            "| Predicted:", predicted_class
        )

        if predicted_class == actual_class:
            correct += 1
        else:
            misclassified.append(
                (image_path, actual_class, predicted_class)
            )

print("Total Test Images:", total)
print("Correct Predictions:", correct)

if total > 0:
    print("Test Accuracy:", round(correct / total * 100, 2), "%")

test_bag1.jpg | Actual: bag | Predicted: fashion_dataset
test_bag2.jpg | Actual: bag | Predicted: test_images
test_shoe1.jpg | Actual: shoe | Predicted: test_images
test_shoe2.jpg | Actual: shoe | Predicted: test_images
test_tshirt1.jpg | Actual: tshirt | Predicted: test_images
test_tshirt2.jpg | Actual: tshirt | Predicted: test_images
Total Test Images: 6
Correct Predictions: 0
Test Accuracy: 0.0 %
